# Pipeline3D — TUM RGB-D · text query → 3D object

End-to-end 2D→3D object localisation on a **TUM RGB-D** sequence:

1. **Index** — `load_tum(...)` builds a dataset-agnostic `FrameSet` (RGB + depth +
   poses + intrinsics), which `Indexer.insert(...)` embeds into LanceDB *together
   with* each frame's `depth_path` + `cam2world` and the per-collection calibration.
2. **Query** — `Search2D` runs `embed | retrieve | detect | rerank | project`. The
   final `ProjectTo3D` step back-projects every hit, fuses them with DBSCAN, and
   returns **one** `ProjectedObject` (cleaned point cloud + axis-aligned world box).
3. **Visualise** — k3d renders the scene cloud, the fused object in red, and its
   3D bounding box.

> Swapping to ScanNet is a one-line change — see `Pipeline3D_ScanNet.ipynb`.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import k3d

# ── ensure project root is on sys.path and is the working directory ──────────
PROJECT_ROOT = Path("__file__").resolve().parent.parent
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = Path("__file__").resolve().parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root : {PROJECT_ROOT}")
print(f"Using device : {device}")

## Dataset config

Run **exactly one** sequence cell. Each sets `DATA_ROOT`, `COLLECTION`, and the
TUM per-sequence `INTRINSICS` + `DEPTH_SCALE` (5000 for TUM).

In [ ]:
# ── fr1 / desk ───────────────────────────────────────────────────────────────
DATA_ROOT   = "data/tum/rgbd_dataset_freiburg1_desk"
COLLECTION  = "fr1_desk"
INTRINSICS  = {"fx": 517.3, "fy": 516.5, "cx": 318.6, "cy": 239.5}
DEPTH_SCALE = 5000.0

In [ ]:
# ── fr2 / desk ───────────────────────────────────────────────────────────────
DATA_ROOT   = "data/tum/rgbd_dataset_freiburg2_desk"
COLLECTION  = "fr2_desk"
INTRINSICS  = {"fx": 520.9, "fy": 521.0, "cx": 325.1, "cy": 249.7}
DEPTH_SCALE = 5000.0

In [ ]:
# ── fr3 / long_office_household ───────────────────────────────────────────────
DATA_ROOT   = "data/tum/rgbd_dataset_freiburg3_long_office_household"
COLLECTION  = "fr3_office"
INTRINSICS  = {"fx": 535.4, "fy": 539.2, "cx": 320.1, "cy": 247.6}
DEPTH_SCALE = 5000.0

## Index

`load_tum` associates RGB↔depth↔pose by nearest timestamp and returns a
`FrameSet`. We keep it in `fs` (the 3D viz reuses it for the scene cloud), then
index — skipping if the collection already exists.

In [ ]:
from src.utils.datasets import load_tum
from src.index import Indexer, get_status
from src.utils.db import connect as db_connect
import yaml

cfg = yaml.safe_load(Path("config.yaml").read_text())
db  = db_connect(cfg["indexing"]["db_path"])

fs = load_tum(DATA_ROOT, intrinsics=INTRINSICS, depth_scale=DEPTH_SCALE)
print(f"Collection : {COLLECTION}")
print(f"Frames     : {len(fs):,}  (RGB + depth + pose, all associated)")

if COLLECTION in db.list_tables().tables:
    n_rows = db.open_table(COLLECTION).count_rows()
    print(f"'{COLLECTION}' already exists with {n_rows:,} rows — skipping indexing.")
else:
    print(f"Indexing '{COLLECTION}' …")
    idx = Indexer.from_config("config.yaml")
    job = idx.insert(
        fs.paths, COLLECTION,
        depth_paths=fs.depth_paths, poses=fs.poses,
        intrinsics=fs.intrinsics, depth_scale=fs.depth_scale,
    )
    status = get_status(job.job_id)
    print(f"Done — {status['processed']}/{status['total']} frames")

## Build pipeline (SAM)

In [ ]:
from src.query.pipeline import Search2D

pipeline = Search2D.from_config("config.yaml", detector="sam")
print("SAM pipeline ready (chain: embed | retrieve | detect | rerank | project).")

## Query

In [ ]:
QUERY          = "laptop"   # ← change to anything
TOP_K_RETRIEVE = 10         # candidate pool fetched from LanceDB
TOP_K_FINAL    = 5          # results kept after re-ranking (all are fused in 3D)

print(f"Running pipeline for: '{QUERY}' …")
state = pipeline.invoke(
    query=QUERY,
    collection_id=COLLECTION,
    top_k_retrieve=TOP_K_RETRIEVE,
    top_k_final=TOP_K_FINAL,
)

print(f"Retrieved : {len(state.retrieved)} candidates")
print(f"Detected  : {len(state.detected)} images")
print(f"Final hits: {len(state.results)}")
print(f"Projected : {len(state.projected)} fused 3D object(s)")

## 2D results

In [ ]:
# ── 2D results: SAM mask overlays + boxes (same helper as Pipeline.ipynb) ─────
PALETTE = [
    (220, 50, 47), (38, 139, 210), (42, 161, 152), (133, 153, 0),
    (211, 54, 130), (181, 137, 0), (108, 113, 196), (203, 75, 22),
]
MASK_ALPHA = 110


def annotate_image(img, masks, boxes, scores, label):
    canvas = img.convert("RGBA")
    for i, mask in enumerate(masks or []):
        color = PALETTE[i % len(PALETTE)]
        mask_np = mask.cpu().numpy().astype(np.uint8)
        overlay = np.zeros((*mask_np.shape, 4), dtype=np.uint8)
        overlay[mask_np > 0] = [*color, MASK_ALPHA]
        canvas = Image.alpha_composite(canvas, Image.fromarray(overlay, "RGBA"))
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", size=14)
    except OSError:
        font = ImageFont.load_default()
    for i, (score, box) in enumerate(zip(scores, boxes)):
        color = PALETTE[i % len(PALETTE)]
        x0, y0, x1, y1 = [int(v) for v in box.tolist()]
        draw.rectangle([x0, y0, x1, y1], outline=(*color, 255), width=2)
        draw.text((x0 + 3, max(y0 - 18, 0)), f"{label[:12]} {float(score):.2f}",
                  fill=(255, 255, 255, 255), font=font)
    return canvas.convert("RGB")


n = len(state.results)
fig, axes = plt.subplots(1, n, figsize=(n * 4.2, 4.5), squeeze=False)
for rank, (ax, hit) in enumerate(zip(axes[0], state.results), start=1):
    annotated = annotate_image(Image.open(hit.path).convert("RGB"),
                               hit.masks, hit.boxes, hit.scores, QUERY)
    ax.imshow(np.array(annotated))
    ax.set_title(f"#{rank}  sim={hit.similarity_score:.3f}  det={hit.detection_score:.3f}",
                 fontsize=9)
    ax.axis("off")
plt.suptitle(f"Query: '{QUERY}'  ·  {COLLECTION}  ·  SAM3", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 3D result

`build_scene_cloud(fs, ...)` reconstructs the room from a sample of full frames;
`state.projected[0]` is the fused object. k3d shows scene + object (red) + box.

In [ ]:
from src.utils.geometry import build_scene_cloud, aabb_corners

SCENE_N_FRAMES = 250   # full frames back-projected for the background cloud
VOXEL          = 0.02  # metres — downsample grid for both clouds


def pack_rgb(colors):
    """Pack a uint8 (N, 3) RGB array into k3d's uint32 0xRRGGBB colours."""
    c = np.asarray(colors, dtype=np.uint32)
    return (c[:, 0] << 16) | (c[:, 1] << 8) | c[:, 2]


# Background scene cloud (reusable helper — same back-projection as the object).
scene_pts, scene_col = build_scene_cloud(fs, n_frames=SCENE_N_FRAMES, voxel=VOXEL)
print(f"Scene cloud: {len(scene_pts):,} points")

obj = state.projected[0]   # the single fused object (cleaned cloud + world AABB)
print(f"Object '{QUERY}': {len(obj.points):,} points  ·  bbox=\n{obj.bbox}")

plot = k3d.plot(grid_visible=False, camera_auto_fit=True)
plot += k3d.points(
    scene_pts.astype(np.float32),
    colors=pack_rgb(scene_col) if scene_col is not None else None,
    point_size=VOXEL, shader="flat", name="scene",
)
plot += k3d.points(
    obj.points.astype(np.float32),
    color=0xff0000, point_size=VOXEL * 1.5, shader="flat", name=f"object: {QUERY}",
)
if obj.bbox is not None:
    corners = aabb_corners(obj.bbox).astype(np.float32)
    EDGES = np.array([[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],
                      [0,4],[1,5],[2,6],[3,7]], dtype=np.uint32)
    plot += k3d.lines(corners, indices=EDGES, indices_type="segment",
                      color=0x00ff00, width=0.01, name="bbox")
plot.display()

## Optional — Grounding DINO (box-only) path

Swap the detector to Grounding DINO. It returns **no masks**, only 2D boxes, but
`ProjectTo3D` still produces a tight fused 3D box: every box is back-projected and
DBSCAN keeps the object (consistent across frames) while per-frame background
scatters into noise. Re-run the **3D result** cell after this to view it.

In [ ]:
pipeline_dino = Search2D.from_config("config.yaml", detector="grounding_dino")
state = pipeline_dino.invoke(
    query=QUERY,                       # dot-separated categories work best for DINO
    collection_id=COLLECTION,
    top_k_retrieve=TOP_K_RETRIEVE,
    top_k_final=TOP_K_FINAL,
)
print(f"Final hits: {len(state.results)}  ·  masks present: "
      f"{any(r.masks for r in state.results)}")
print(f"Projected : {len(state.projected)} fused 3D object(s) "
      f"(box back-projection + DBSCAN)")